### Relation Extraction

#### This section covers the relation extraction component of the model and expects the following:

<ol>
<li>Mention Span Vectors</li>
<li>Entity List</li>
<li>Relation label set</li>
</ol>

### Import

In [ ]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence

In [ ]:
def mention_pooling(mention_embeddings, mask, pool_type='mean', attn_scorer=None):
    # mention_embeddings: (K, max_m, 2304), padded
    # mask:               (K, max_m) bool, True = real mention, False = padding
    if pool_type == 'mean':
        m = mask.unsqueeze(-1).float()                                   # (K, max_m, 1)
        return (mention_embeddings * m).sum(dim=1) / m.sum(dim=1)        # (K, 2304)

    elif pool_type == 'log-sum-exp':
        x = mention_embeddings.masked_fill(~mask.unsqueeze(-1), float('-inf'))
        return torch.logsumexp(x, dim=1)                                 # (K, 2304)

    elif pool_type == 'attention':
        if attn_scorer is None:
            raise ValueError("attention pooling needs attn_scorer = nn.Linear(2304, 1)")
        scores = attn_scorer(mention_embeddings).squeeze(-1)             # (K, max_m)
        scores = scores.masked_fill(~mask, float('-inf'))
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)             # (K, max_m, 1)
        return (weights * mention_embeddings).sum(dim=1)                 # (K, 2304)

    else:
        raise ValueError(f"Unknown pooling type: {pool_type}")

In [ ]:
class RelationExtractionModel(nn.Module):
    def __init__(self, input_dim = 4608,hidden_dim=150, label_dim=768,embedding_dim=768,dropout_rate=0.3, pool_type='mean'):
        super(RelationExtractionModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, embedding_dim)
        self.relation_projection = nn.Linear(label_dim, embedding_dim)
        self.dropout = nn.Dropout(dropout_rate)
        self.relu = nn.ReLU()
        self.pool_type = pool_type                     # 'mean' | 'log-sum-exp' | 'attention'
        if pool_type == 'attention':
            self.attn_scorer = nn.Linear(2304, 1)
    
    def forward(self, span_vecs, mention_to_entity, label_embs):
        # span_vecs: (M,2304) Mention span vectors from part 1
        # mention_to_entity: (M,) LongTensor, mapping from mention index to entity index
        # label_embs: (R,768) Label embeddings for each entity
        
        if span_vecs.size(0) == 0:
            return span_vecs.new_zeros(0, label_embs.size(0)), span_vecs.new_zeros(0, 2, dtype=torch.long)
        
        # 1. Group mentions by entity (K, max_m, 2304) + mask
        K = int(mention_to_entity.max()) + 1
        groups = [span_vecs[mention_to_entity == k] for k in range(K)]
        mention_embeddings = pad_sequence(groups, batch_first=True)  # (K, max_m, 2304)
        lengths = torch.tensor([g.size(0) for g in groups], device=span_vecs.device)
        mask = torch.arange(mention_embeddings.size(1), device=span_vecs.device).unsqueeze(0) < lengths.unsqueeze(1)

        # 2. Pool mention embeddings to get entity embeddings (K, 2304)
        entity_embeddings = mention_pooling(mention_embeddings, mask, self.pool_type,getattr(self, 'attn_scorer', None))  # (K, 2304)
        
        #3. All ordered pairs (source != target)
        idx = torch.arange(K, device=span_vecs.device)
        pairs = torch.cartesian_prod(idx, idx)  # (K*K, 2)
        pairs = pairs[pairs[:,0] != pairs[:,1]]
        if pairs.numel() == 0:
            return span_vecs.new_zeros(0, label_embs.size(0)), pairs
        
        
        # 4. Pair representation: concat(source, target) (P, 4608)
        pair_rep = torch.cat([entity_embeddings[pairs[:,0]], entity_embeddings[pairs[:,1]]], dim=1)  # (P, 4608)
        
        #5. pair_proj
        pair_proj = self.relu(self.fc1(pair_rep))  # (P, hidden_dim)
        pair_proj = self.dropout(pair_proj)
        pair_proj = self.fc2(pair_proj)  # (P, 768)
        
        #6. label_proj
        label_proj = self.relation_projection(label_embs)  # (R, 768)
        
        #7. Dot product to get scores (P, R)
        logits = pair_proj @ label_proj.T
        
        return logits, pairs
    
    @torch.no_grad()
    def predict(self,logits, pairs, entity_names, relation_names, threshold=0.5):
        # logits:         (P, R)
        # pairs:          (P, 2)  source and target entity ids
        # entity_names:   list of K strings, e.g. ['Maya', 'Northstar Labs']
        # relation_names: list of R strings, e.g. ['founder of', 'works for', 'located in']
        # thresholds:     float, or tensor (R,) for one threshold per relation  
        
        probs = torch.sigmoid(logits)  # (P, R)
        if not torch.is_tensor(threshold):
            threshold = torch.as_tensor(threshold, dtype=probs.dtype, device=probs.device)
            if threshold.dim() == 0:
                threshold = threshold.expand(probs.size(1))
        
        matches = (probs > threshold.unsqueeze(0)).nonzero()
        
        triples = []
        for pair, relation in matches.tolist():
            src, tgt = pairs[pair].tolist()
            triples.append({
                "source": entity_names[src],
                "relation": relation_names[relation],
                "target": entity_names[tgt],
                "score": round(probs[pair, relation].item(), 3)
            })
        return triples